# 10 -- Pelne obliczenia produkcji (integracja modeli)

Ten notebook **zlozy w jedno** wszystkie modele z notebookow 05-09:

1. **Hydrologia** (nb 04, 05) -- sredni rok uporzadkowany Q i H
2. **Straty hydrauliczne** (nb 06) -- `loss_fns`, spad netto $H_{net}(Q)$
3. **Turbina** (nb 07) -- typ, n_turbin, $\eta_t(Q/Q_{des})$, rozdzial przeplywu
4. **Generator** (nb 08) -- $\eta_g(P/P_n)$ zalezne od obciazenia
5. **Koszty** (nb 09) -- inwestycja, O&M, NPV, LCOE

Integrator: `src/production.py` → `compute_power()`.

**Cel:** porownac realny model z uproszczonym $\eta_{total} = const$ z nb 05.

---

## Modul `src/production.py`

**Prompt do LLM tworzacy ten modul:**
> *"Stworz modul src/production.py jako warstwe integrujaca: funkcja
> `compute_power(Q_sorted, H_gross, Q_design, n_turbines, turbine_type, loss_fns,
> H_design, gen_k_cu, gen_k_fe, gen_k_mech, rho, g)` ktora dla kazdego dnia
> sredniego roku uporzadkowanego: (1) rozdziela przeplyw na turbiny,
> (2) liczy straty hydrauliczne, (3) sprawnosc turbiny, (4) moc na walu,
> (5) sprawnosc generatora (per-unit), (6) moc elektryczna, (7) energie dobowa.
> Zwraca DataFrame z wszystkimi kolumnami posrednimi.
> Dodatkowo funkcja `compute_power_constant_eta()` do replikacji uproszczenia z nb05."*

**Funkcje w module:**
- `compute_power(...)` -- pelny model (losses + var η_t + var η_g)
- `compute_power_constant_eta(...)` -- model staly (jak w nb 05) dla porownania
- `annual_energy(df)`, `capacity_factor(df, P_rated)`, `operating_hours(df)`, `average_efficiency(df)` -- metryki podsumowujace

## Konfiguracja

**Prompt do LLM:**
> *"Napisz kod konfiguracji notebooka 10: zaladuj `compute_power` i metryki
> z `src/production`, `TURBINE_CATALOG` z `src/turbine`, funkcje strat
> z `src/losses`, funkcje kosztow z `src/costs`, oraz hydrologie
> (`average_sorted_year`, `filter_incomplete_years`, `transfer_flow_between`)
> i `imgw_data.load_processed`."*

**Uzyte moduly/funkcje:** `src.production`, `src.turbine`, `src.losses`, `src.costs`, `src.hydrology`, `src.imgw_data`, `src.watershed`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.imgw_data import load_processed
from src.hydrology import (
    average_sorted_year, filter_incomplete_years,
)
from src.watershed import get_station_area, find_gauge, get_catchment_area

from src.losses import (
    trash_rack_loss, pipe_friction_loss, minor_loss, net_head,
)
from src.turbine import TURBINE_CATALOG, runner_diameter, rotational_speed, synchronous_speed
from src.production import (
    compute_power, compute_power_constant_eta,
    annual_energy, capacity_factor, operating_hours, average_efficiency,
)
from src.costs import total_investment, economic_analysis, EUR_PLN_RATE

pd.set_option('display.max_columns', 15)
print('Moduly zaladowane.')

---
## Krok 1: Sredni rok uporzadkowany Q i H

Powtarzamy w skrocie procedure z nb 05: interpolujemy przeplyw z dwoch wodowskazow
do lokalizacji MEW, budujemy sredni rok uporzadkowany dla Q i dla stanu wody.

**Prompt do LLM:**
> *"Wczytaj dane z `data/processed/daily_hydro_clean.parquet`. Zinterpoluj przeplyw
> Q_mew z gauges Brzeg Dolny (upstream) i Malczyce (downstream) metoda potegowa
> wg pol zlewni. Wyznacz sredni rok uporzadkowany Q_sorted oraz H_net_sorted
> z modelu spadu (Hsa = H_stage + level_avg - level_day)."*

**Uzyte funkcje:** `load_processed`, `filter_incomplete_years`, `average_sorted_year`, `get_station_area`, `find_gauge`, `get_catchment_area`.

In [ ]:
# Wczytaj dane
df = load_processed('../data/processed/daily_hydro_clean.parquet')

STATION_UP = '151160170'    # Brzeg Dolny (upstream)
STATION_DOWN = '151160150'  # Malczyce (downstream)

A_UP = get_station_area(STATION_UP)
A_DOWN = get_station_area(STATION_DOWN)
gauge_up = find_gauge(STATION_UP)
gauge_down = find_gauge(STATION_DOWN)
lat_mew = (gauge_up['lat'] + gauge_down['lat']) / 2
lng_mew = (gauge_up['lng'] + gauge_down['lng']) / 2
A_TARGET = get_catchment_area(lat_mew, lng_mew, label='MEW')

# Interpolacja przeplywu metoda potegowa (per-day n; szczegoly w nb 05)
df_up = df[df['station_id'] == STATION_UP][['date', 'discharge_m3s', 'water_level_cm']].rename(
    columns={'discharge_m3s': 'Q_up', 'water_level_cm': 'level_up'})
df_down = df[df['station_id'] == STATION_DOWN][['date', 'discharge_m3s', 'water_level_cm']].rename(
    columns={'discharge_m3s': 'Q_down', 'water_level_cm': 'level_down'})
df_mew = df_up.merge(df_down, on='date', how='inner').dropna()
ln_ratio = np.log(A_DOWN / A_UP)
n_exp = np.log(df_mew['Q_down'] / df_mew['Q_up']) / ln_ratio
df_mew['Q_mew'] = df_mew['Q_up'] * (A_TARGET / A_UP) ** n_exp

# Sredni rok uporzadkowany Q
df_q = pd.DataFrame({'station_id': 'MEW', 'date': df_mew['date'], 'discharge_m3s': df_mew['Q_mew']})
df_q = filter_incomplete_years(df_q, 'MEW', min_completeness=0.95)
avg_year_q = average_sorted_year(df_q, 'MEW')
Q_sorted = avg_year_q['mean'].values

# Sredni rok uporzadkowany stanu wody (wodowskaz dolny — Malczyce)
df_lvl = filter_incomplete_years(df, STATION_DOWN, min_completeness=0.95)
avg_year_lvl = average_sorted_year(df_lvl, STATION_DOWN, column='water_level_cm')
level_sorted = avg_year_lvl['mean'].values / 100.0  # m
level_avg = (df_lvl[df_lvl['station_id'] == STATION_DOWN]['water_level_cm'].dropna() / 100.0).mean()

# Spad netto (gross): zmienia sie wg odchylenia stanu wody od sredniego
H_STAGE = 6.0  # spad brutto na stopniu wodnym
H_gross = np.maximum(H_STAGE + (level_avg - level_sorted), 0.0)

print(f'Zlewnia MEW = {A_TARGET:.0f} km2')
print(f'Sredni rok uporzadkowany: {len(Q_sorted)} dni')
print(f'  Q: min={Q_sorted.min():.1f}, mean={Q_sorted.mean():.1f}, max={Q_sorted.max():.1f} m3/s')
print(f'  H_gross: min={H_gross.min():.2f}, max={H_gross.max():.2f} m (H_stage={H_STAGE} m)')

---
## Krok 2: Konfiguracja drogi wodnej (straty hydrauliczne)

Definiujemy `loss_fns` -- liste funkcji $Q \\to \\Delta H$. Tutaj instalacja
rurociagowa typowa dla niskiego spadu (jak w nb 06).

**Prompt do LLM:**
> *"Zdefiniuj `loss_fns` dla naszej drogi wodnej: krata, krotki rurociag (D=2.5m,
> L=15m), wlot zaokraglony, jedno kolano, zasuwa motylkowa, spirala, rura ssawna."*

**Uzyte funkcje:** `trash_rack_loss`, `pipe_friction_loss`, `minor_loss`.

In [ ]:
D_pipe = 2.5
A_pipe = np.pi * D_pipe**2 / 4
A_spiral = 4.0  # m2
A_runner = 1.5  # m2 (wyjscie z wirnika)

loss_fns = [
    lambda Q: trash_rack_loss(Q, A_rack=8.0, bar_width=0.012, bar_spacing=0.05),
    lambda Q: pipe_friction_loss(Q, D=D_pipe, L=15.0, k_s=0.001),
    lambda Q: minor_loss(Q, A_pipe, xi=0.15),   # wlot zaokraglony
    lambda Q: minor_loss(Q, A_pipe, xi=0.25),   # kolano
    lambda Q: minor_loss(Q, A_pipe, xi=0.30),   # zasuwa
    lambda Q: minor_loss(Q, A_spiral, xi=0.10), # spirala turbiny
    lambda Q: minor_loss(Q, A_runner, xi=0.25), # rura ssawna
]

# Wykres H_gross vs H_net
Q_test = np.linspace(5, 60, 100)
H_net_test = net_head(H_STAGE, Q_test, loss_fns)

fig = go.Figure()
fig.add_hline(y=H_STAGE, line_dash='dash', line_color='gray',
    annotation_text=f'H_brutto = {H_STAGE} m')
fig.add_trace(go.Scatter(x=Q_test, y=H_net_test, mode='lines',
    line=dict(color='royalblue', width=2), name='H_netto(Q)'))
fig.update_layout(
    title='Spad brutto vs netto (droga wodna MEW)',
    xaxis_title='Q [m3/s]', yaxis_title='H [m]',
    height=400, hovermode='x unified',
)
fig.show()

print(f'Strata przy Q=30 m3/s: {H_STAGE - float(net_head(H_STAGE, 30.0, loss_fns)):.3f} m')

---
## Krok 3: Wybor turbiny i punkt projektowy

Wybieramy typ turbiny, liczbe agregatow i $Q_{design}$ na turbine.
Punkt projektowy to **dzien instalacyjny** na krzywej uporzadkowanej.
Tutaj startujemy od $d_{install} = 70$ (~19% przekroczenia) — taki sam jak w arkuszu WPE_2.xlsm.

**Prompt do LLM:**
> *"Skonfiguruj wybor: Kaplan, 2 turbiny, dzien instalacyjny 70. Oblicz
> Q_design na turbine i wymiarowanie wirnika (D1, n)."*

**Uzyte funkcje:** `TURBINE_CATALOG`, `runner_diameter`, `rotational_speed`, `synchronous_speed`.

In [ ]:
# Konfiguracja turbinowa
turbine_type = TURBINE_CATALOG['kaplan']
N_TURBINES = 2
INSTALL_DAY = 70

# Q_design (na turbine): calkowite Q na dniu instalacyjnym / liczba turbin
Q_total_design = float(Q_sorted[INSTALL_DAY - 1])
Q_design = Q_total_design / N_TURBINES
H_design = float(H_gross[INSTALL_DAY - 1])  # spad na dniu projektowym

# Wymiarowanie wirnika (jednego)
D1 = runner_diameter(Q_design, H_design, turbine_type)
n_raw = rotational_speed(H_design, D1, turbine_type)
n_sync, poles = synchronous_speed(n_raw)

print(f'Turbina: {turbine_type.name_pl}, liczba: {N_TURBINES}')
print(f'Dzien instalacyjny: {INSTALL_DAY} ({INSTALL_DAY/365*100:.1f}% przekroczenia)')
print(f'Q_total_design = {Q_total_design:.2f} m3/s')
print(f'Q_design/turbine = {Q_design:.2f} m3/s')
print(f'H_design = {H_design:.2f} m')
print(f'D1 = {D1:.3f} m')
print(f'n = {n_raw:.0f} obr/min -> n_sync = {n_sync:.0f} obr/min ({poles} biegunow)')

---
## Krok 4: Pelne obliczenie produkcji

Teraz puszczamy wszystko przez `compute_power()` -- integrator wykonuje
po kolei: rozdzial → straty → η_t(Q) → moc na walu → η_g(P/Pn) (per-unit) → energia.

**Prompt do LLM:**
> *"Wywolaj `compute_power(Q_sorted, H_gross, Q_design, n_turbines=2, turbine_type=Kaplan,
> loss_fns=loss_fns, H_design=H_design)`. Wyswietl pierwsze 5 wierszy DataFrame
> oraz metryki: energia roczna [MWh], capacity factor, dni pracy, srednia η_t i η_g."*

**Uzyte funkcje:** `compute_power`, `annual_energy`, `capacity_factor`, `operating_hours`, `average_efficiency`.

In [ ]:
# Pelne obliczenie produkcji
result = compute_power(
    Q_sorted=Q_sorted,
    H_gross=H_gross,
    Q_design=Q_design,
    n_turbines=N_TURBINES,
    turbine_type=turbine_type,
    loss_fns=loss_fns,
    H_design=H_design,
)

# Metryki
E_MWh = annual_energy(result)
P_rated_unit_kW = 998 * 9.81 * Q_design * H_design * turbine_type.eta_peak / 1000.0
P_rated_total_kW = P_rated_unit_kW * N_TURBINES
CF = capacity_factor(result, P_rated_total_kW)
oh = operating_hours(result)
eta_avg = average_efficiency(result)

print(f'=== Pelny model produkcji ===')
print(f'  Moc znamionowa (caly zaklad): {P_rated_total_kW:.0f} kW '
      f'({N_TURBINES} x {P_rated_unit_kW:.0f} kW)')
print(f'  Energia roczna:               {E_MWh:.0f} MWh/rok')
print(f'  Wspolczynnik wykorzystania:   {CF*100:.1f}%')
print(f'  Dni pracy:                    {oh} / {len(Q_sorted)}')
print(f'  Srednia η_t (wazona energia): {eta_avg["eta_t_avg"]:.3f}')
print(f'  Srednia η_g (wazona energia): {eta_avg["eta_g_avg"]:.3f}')
print(f'  Srednia η_t · η_g:            {eta_avg["eta_total_avg"]:.3f}')
print()
print('Pierwsze 5 wierszy DataFrame:')
result.head()

### Wykres produkcji vs procent przekroczenia

**Prompt do LLM:**
> *"Narysuj 4 wykresy w siatce 2x2: (1) Q dostepne i Q uzyte, (2) sprawnosci η_t i η_g,
> (3) liczba aktywnych turbin, (4) moc elektryczna. Wszystko vs procent przekroczenia."*

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=['Q [m3/s] (dostepne vs uzyte)', 'Sprawnosci η_t, η_g',
                    'Liczba aktywnych turbin', 'Moc elektryczna [kW]'],
    vertical_spacing=0.13, horizontal_spacing=0.10)

pct = result['pct'].values

# (1) Q
fig.add_trace(go.Scatter(x=pct, y=result['Q_avail'], mode='lines',
    name='Q dostepne', line=dict(color='gray', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=pct, y=result['Q_used'], mode='lines',
    name='Q uzyte', line=dict(color='royalblue', width=2)), row=1, col=1)

# (2) sprawnosci
fig.add_trace(go.Scatter(x=pct, y=result['eta_t'], mode='lines',
    name='η_t', line=dict(color='firebrick', width=2)), row=1, col=2)
fig.add_trace(go.Scatter(x=pct, y=result['eta_g'], mode='lines',
    name='η_g', line=dict(color='green', width=2)), row=1, col=2)

# (3) n_active
fig.add_trace(go.Scatter(x=pct, y=result['n_active'], mode='lines',
    name='n_active', line=dict(color='purple', width=2, shape='hv'),
    showlegend=False), row=2, col=1)

# (4) P_el
fig.add_trace(go.Scatter(x=pct, y=result['P_el_kW'], mode='lines',
    name='P_el [kW]', fill='tozeroy',
    line=dict(color='darkorange', width=1.5), showlegend=False), row=2, col=2)

fig.update_xaxes(title_text='Prawdopodobienstwo przekroczenia [%]', row=2, col=1)
fig.update_xaxes(title_text='Prawdopodobienstwo przekroczenia [%]', row=2, col=2)
fig.update_yaxes(title_text='Q [m3/s]', row=1, col=1)
fig.update_yaxes(title_text='η [-]', range=[0, 1], row=1, col=2)
fig.update_yaxes(title_text='liczba turbin', row=2, col=1)
fig.update_yaxes(title_text='P [kW]', row=2, col=2)
fig.update_layout(height=700, hovermode='x unified',
    title='Pelna produkcja MEW — Kaplan 2×{:.0f} kW'.format(P_rated_unit_kW))
fig.show()

---
## Krok 5: Porownanie z modelem stalym (nb 05)

Zobaczmy ile wynosi roznica w energii rocznej miedzy:
- **(A) Pelny model** (straty + var η_t + var η_g) — to co liczymy w tym notebooku,
- **(B) Model staly** — jak w nb 05: brak strat, $\\eta_{total} = 0.848$.

**Prompt do LLM:**
> *"Porownaj energie roczna i moc dla dwoch modeli: pelnego i stalego.
> Pokaz tabelke z roznica i wykres mocy obu modeli."*

**Uzyte funkcje:** `compute_power_constant_eta`, `annual_energy`.

In [ ]:
# Model staly (nb 05): brak strat, eta_total = 0.848
result_simple = compute_power_constant_eta(
    Q_sorted=Q_sorted,
    H_gross=H_gross,
    Q_design=Q_total_design,  # tutaj Q_design oznacza CALKOWITE (jak w nb 05)
    n_turbines=N_TURBINES,
    Q_min_fraction=turbine_type.Q_ratio_min,
    eta_total=0.848,
)

E_simple = annual_energy(result_simple)

# Tabelka porownawcza
print('=== Porownanie modeli ===')
print(f'                              Pelny       Staly (nb05)   Roznica')
print(f'  Energia roczna [MWh]:      {E_MWh:>7.0f}     {E_simple:>7.0f}        {E_MWh - E_simple:+.0f} ({(E_MWh/E_simple - 1)*100:+.1f}%)')
print(f'  Dni pracy:                 {oh:>7d}     {(result_simple["P_kW"] > 0).sum():>7d}')
print(f'  Sprawnosc dziala (avg):    {eta_avg["eta_total_avg"]:>7.3f}     {0.848:>7.3f}')

# Wykres mocy
fig = go.Figure()
fig.add_trace(go.Scatter(x=result['pct'], y=result['P_el_kW'],
    mode='lines', name='Pelny model (P_el)',
    line=dict(color='royalblue', width=2.5)))
fig.add_trace(go.Scatter(x=result_simple['pct'], y=result_simple['P_kW'],
    mode='lines', name='Model staly (P)',
    line=dict(color='gray', width=2, dash='dash')))
fig.update_layout(
    title='Moc na krzywej uporzadkowanej — pelny vs staly model',
    xaxis_title='Prawdopodobienstwo przekroczenia [%]',
    yaxis_title='P [kW]', height=450, hovermode='x unified',
)
fig.show()

print()
print('Wnioski:')
print('  - Pelny model uwzglednia straty hydrauliczne (~kilka % spadu)')
print('    i spadek sprawnosci przy czesciowym obciazeniu')
print('  - W rezultacie energia roczna jest **nizsza** niz w modelu stalym')
print('  - Roznica zwykle 5-15% — zalezy od konfiguracji drogi wodnej i przeplywow')

---
## Krok 6: Optymalizacja punktu projektowego z pelnym modelem

W nb 05 optymalizowalismy dzien instalacyjny ze stalym $\\eta$. Tutaj robimy
to samo z **pelnym modelem** — i porownujemy obie krzywe.

**Prompt do LLM:**
> *"Dla dni instalacyjnych 20..200 (krok 5) wywolaj `compute_power` i zbierz
> energie roczna. Porownaj z modelem stalym. Wyznacz optimum w obu przypadkach."*

**Uzyte funkcje:** `compute_power`, `compute_power_constant_eta`, `annual_energy`.

In [ ]:
# Sweep dnia instalacyjnego — wolniej niz nb 05 (compute_power liczy duzo)
# wiec idziemy z wiekszym krokiem.
days = np.arange(20, 201, 5)
rows = []
for d in days:
    Q_total = float(Q_sorted[d - 1])
    Q_d = Q_total / N_TURBINES
    H_d = float(H_gross[d - 1])

    # Pelny model
    r_full = compute_power(Q_sorted, H_gross, Q_d, N_TURBINES,
                           turbine_type, loss_fns=loss_fns, H_design=H_d)
    E_full = annual_energy(r_full)
    P_inst = 998 * 9.81 * Q_d * H_d * turbine_type.eta_peak / 1000.0 * N_TURBINES

    # Model staly (nb05)
    r_simple = compute_power_constant_eta(
        Q_sorted, H_gross, Q_total, N_TURBINES,
        Q_min_fraction=turbine_type.Q_ratio_min, eta_total=0.848)
    E_simple_d = annual_energy(r_simple)

    rows.append({
        'install_day': d,
        'install_pct': round(d / len(Q_sorted) * 100, 1),
        'Q_total_design': Q_total,
        'P_installed_kW': P_inst,
        'E_full_MWh': E_full,
        'E_simple_MWh': E_simple_d,
        'roznica_pct': (E_full / E_simple_d - 1) * 100 if E_simple_d > 0 else 0,
    })
df_sweep = pd.DataFrame(rows)

best_full = df_sweep.loc[df_sweep['E_full_MWh'].idxmax()]
best_simple = df_sweep.loc[df_sweep['E_simple_MWh'].idxmax()]
print(f'Optimum pelnego modelu:   dzien {int(best_full["install_day"])} '
      f'({best_full["install_pct"]:.0f}%), E = {best_full["E_full_MWh"]:.0f} MWh, '
      f'P = {best_full["P_installed_kW"]:.0f} kW')
print(f'Optimum modelu stalego:   dzien {int(best_simple["install_day"])} '
      f'({best_simple["install_pct"]:.0f}%), E = {best_simple["E_simple_MWh"]:.0f} MWh, '
      f'P = {best_simple["P_installed_kW"]:.0f} kW')

# Wykres
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_sweep['install_pct'], y=df_sweep['E_full_MWh'],
    mode='lines', name='Pelny model', line=dict(color='royalblue', width=2.5)))
fig.add_trace(go.Scatter(x=df_sweep['install_pct'], y=df_sweep['E_simple_MWh'],
    mode='lines', name='Model staly (nb05)',
    line=dict(color='gray', width=2, dash='dash')))
fig.add_vline(x=best_full['install_pct'], line_color='royalblue', line_dash='dot',
    annotation_text=f'opt pelny {best_full["install_pct"]:.0f}%')
fig.add_vline(x=best_simple['install_pct'], line_color='gray', line_dash='dot',
    annotation_text=f'opt staly {best_simple["install_pct"]:.0f}%')
fig.update_layout(
    title='Energia roczna vs punkt projektowy — porownanie modeli',
    xaxis_title='Prawdopodobienstwo przekroczenia [%]',
    yaxis_title='E [MWh/rok]', height=450, hovermode='x unified',
)
fig.show()

---
## Krok 7: Analiza ekonomiczna dla optimum

Dla optymalnego punktu z pelnego modelu liczymy inwestycje, NPV, LCOE i okres zwrotu.

**Prompt do LLM:**
> *"Dla optymalnego punktu (dzien {best_full['install_day']}) policz inwestycje
> `total_investment()` i ekonomie `economic_analysis()` przy cenie 106.5 EUR/MWh
> (= 490 PLN/MWh). Wyswietl tabele wynikow z konwersja na PLN."*

**Uzyte funkcje:** `total_investment`, `economic_analysis`, `EUR_PLN_RATE`.

In [ ]:
# Optimum z pelnego modelu
d_opt = int(best_full['install_day'])
Q_opt_total = float(Q_sorted[d_opt - 1])
Q_opt = Q_opt_total / N_TURBINES
H_opt = float(H_gross[d_opt - 1])
P_unit_opt = 998 * 9.81 * Q_opt * H_opt * turbine_type.eta_peak / 1000.0
E_opt = best_full['E_full_MWh']

# Inwestycja
inv = total_investment(
    P_kW=P_unit_opt,
    H=H_opt,
    n_turbines=N_TURBINES,
    turbine_type='kaplan',
    plant_type='run_of_river_small',
)

# Ekonomia
econ = economic_analysis(
    energy_mwh=E_opt,
    investment_eur=inv['total_eur'],
    energy_price_eur_mwh=106.5,  # 490 PLN/MWh / 4.6
    om_fraction=0.025,
    discount_rate=0.06,
    lifetime_years=40,
)

# Wyniki
print(f'=== Analiza ekonomiczna — pelny model, dzien optymalny {d_opt} ===')
print(f'  Moc zainstalowana: {inv["P_total_kW"]:.0f} kW ({N_TURBINES} x {P_unit_opt:.0f} kW)')
print(f'  Energia roczna:    {E_opt:.0f} MWh/rok')
print(f'  Capacity factor:   {E_opt * 1000 / (inv["P_total_kW"] * 8760) * 100:.1f}%')
print()
print(f'  Inwestycja:        {inv["total_eur"]:>12,.0f} EUR = {inv["total_eur"] * EUR_PLN_RATE:>14,.0f} PLN')
print(f'    EM:              {inv["em_eur"]:>12,.0f} EUR ({inv["em_eur"] / inv["total_eur"] * 100:.0f}%)')
print(f'    Roboty bud.:     {inv["civil_eur"]:>12,.0f} EUR ({inv["civil_eur"] / inv["total_eur"] * 100:.0f}%)')
print(f'    Przylacze:       {inv["grid_eur"]:>12,.0f} EUR ({inv["grid_eur"] / inv["total_eur"] * 100:.0f}%)')
print(f'    Inzynieria:      {inv["engineering_eur"]:>12,.0f} EUR ({inv["engineering_eur"] / inv["total_eur"] * 100:.0f}%)')
print(f'  EUR/kW:            {inv["total_per_kw_eur"]:>12,.0f}')
print()
print(f'  Przychod roczny:   {econ["annual_revenue_eur"]:>12,.0f} EUR/rok = '
      f'{econ["annual_revenue_eur"] * EUR_PLN_RATE:>14,.0f} PLN/rok')
print(f'  O&M roczne:        {econ["annual_om_eur"]:>12,.0f} EUR/rok')
print(f'  Dochod netto:      {econ["net_annual_eur"]:>12,.0f} EUR/rok')
print()
print(f'  Okres zwrotu:      {econ["payback_years"]:>12.1f} lat')
print(f'  NPV (r=6%, T=40):  {econ["npv_eur"]:>12,.0f} EUR')
print(f'  LCOE:              {econ["lcoe_eur_mwh"]:>12.1f} EUR/MWh = {econ["lcoe_eur_mwh"] * EUR_PLN_RATE:.0f} PLN/MWh')

---
## Podsumowanie

W tym notebooku **zlozyliśmy w jedno** wszystkie wczesniejsze modele:

| Krok | Co | Z notebooka |
|------|-----|-------------|
| Krok 1 | Q_sorted, H_gross | 04, 05 |
| Krok 2 | `loss_fns` (straty hydrauliczne) | 06 |
| Krok 3 | Wybor turbiny i wymiarowanie | 07 |
| Krok 4 | `compute_power()` → produkcja | 08 (η_g), `src/production.py` (integrator) |
| Krok 5 | Porownanie pelny vs staly model | 05 vs 06-08 |
| Krok 6 | Optymalizacja punktu projektowego | 05 (rozszerzone) |
| Krok 7 | Analiza ekonomiczna | 09 |

### Kluczowe roznice pelnego modelu vs uproszczonego z nb 05

- **Spad netto** $H_{net}(Q) < H_{brutto}$ — straty rosna z $Q^2$, najwieksze przy szczytach,
- **Sprawnosc turbiny** $\\eta_t$ spada przy czesciowym obciazeniu (Kaplan zachowuje >90% w szerokim zakresie),
- **Sprawnosc generatora** $\\eta_g$ ma maksimum przy ~75-80% $P_n$, **nie 100%**,
- **Rozdzial przeplywu** — przy 2 turbinach lepsza praca przy szczytach (mniej spilla).

### Co dalej (sugestie dla studentow)

1. Zmien typ turbiny (np. propeller, crossflow) i porownaj — niektore odpadaja przez $H_{range}$.
2. Zmien $N_{turbines}$ z 2 na 1, 3, 4 — zobacz wplyw na CF i NPV.
3. Wymien jeden komponent strat (np. usun rurociag — robic kanal otwarty).
4. Sprawdz wrazliwosc na cene energii (90, 110, 130 EUR/MWh).
5. Wprowadz wlasna konfiguracje drogi wodnej i policz wlasny wariant.

**Wszystkie powyzsze zmiany sa kilkulinijkowymi modyfikacjami w odpowiednich krokach.**